# Fire Data Analysis

This notebook analyzes and displays fire data from 1972-2022:
- **FEUX_CONT_SIMP_1972_2022.shp**: Fire data from 1972-2022

The goal is to load the data, check its availability, allow the user
to specify a year for analysis, and visualize the fire data for that year.

## Section 1: Import Required Libraries

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
from folium import plugins
import json
from IPython.display import IFrame, HTML
import fiona
import warnings
warnings.filterwarnings('ignore')
import os
from dbfread import DBF
import matplotlib.pyplot as plt

# Set up display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

## Section 2: Load Dataset

In [ ]:
## Section 2: Load Dataset and Check Availability

# Check if fire contours data exists in the new directory structure
fire_contours_path = '../data/shapefiles/fire_contours'
shapefile_path = f'{fire_contours_path}/FEUX_CONT_SIMP_1972_2022.shp'

print("Checking for fire contours data in new directory structure...")
if not os.path.exists(shapefile_path):
    print(f"ERROR: Fire contours data not found at {shapefile_path}")
    print("Please run 'download_geopackages.ipynb' first to download all required geopackages.")
    raise FileNotFoundError(f"Fire contours shapefile not found: {shapefile_path}")
else:
    print(f"✓ Fire contours data found at {shapefile_path}")

# Load the FEUX_CONT_SIMP_1972_2022 dataset
print("\nLoading FEUX_CONT_SIMP_1972_2022.shp...")
with fiona.open(shapefile_path) as src:
    features = [feature for feature in src]
    feux_1972_2022 = gpd.GeoDataFrame.from_features(features, crs=src.crs)
    
    # Get detailed schema information
    print(f"\n  - Records: {len(feux_1972_2022)}")
    print(f"  - Geometry type: {feux_1972_2022.geometry.geom_type.unique()}")
    print(f"  - CRS: {feux_1972_2022.crs}")
    print(f"  - Columns: {list(feux_1972_2022.columns)}")
    
    # Get field information from fiona schema
    print(f"\n  Detailed field information:")
    for field, field_type in src.schema['properties'].items():
        print(f"    - {field}: {field_type}")

# Read DBF file for additional schema information
print("\n" + "=" * 80)
print("DBF FILE SCHEMA INFORMATION")
print("=" * 80)

dbf_path = f'{fire_contours_path}/FEUX_CONT_SIMP_1972_2022.dbf'
print(f"\nFEUX_CONT_SIMP_1972_2022.dbf schema:")
table1 = DBF(dbf_path)
print(f"  - Total records: {len(table1)}")
print(f"  - Field definitions:")
for field in table1.fields:
    print(f"    - {field.name}: {field.type}, length={field.length}, decimal={field.decimal_count}")
        
# Check for metadata files
print("\n" + "=" * 80)
print("CHECKING FOR METADATA FILES")
print("=" * 80)

metadata_files = []
for file in os.listdir(fire_contours_path):
    if file.endswith('.xml') or file.endswith('.cpg') or file.endswith('.prj'):
        metadata_files.append(file)

if metadata_files:
    print(f"\nFound metadata files: {metadata_files}")
    
    # Try to read XML metadata if available
    for xml_file in metadata_files:
        if xml_file.endswith('.xml'):
            xml_path = os.path.join(fire_contours_path, xml_file)
            try:
                with open(xml_path, 'r') as f:
                    xml_content = f.read()
                print(f"\n{xml_file} content (first 500 chars):")
                print(xml_content[:500])
            except Exception as e:
                print(f"\nCould not read {xml_file}: {e}")
else:
    print("\nNo additional metadata files found (.xml, .cpg, .prj)")

# Display sample data to infer column meanings
print("\n" + "=" * 80)
print("SAMPLE DATA FOR COLUMN INTERPRETATION")
print("=" * 80)

print("\nFEUX_CONT_SIMP_1972_2022.shp - First 3 rows:")
display(feux_1972_2022.head(3))

## Section 4: Specify Analysis Year

In [ ]:
# Allow user to specify the year for analysis
print("=" * 80)
print("SPECIFY ANALYSIS YEAR")
print("=" * 80)

# Set the year for analysis (change this value to analyze different years)
analysis_year = 2021  # Modify this to analyze a different year (1972-2022)

# Validate the year is within the available range
min_year = int(feux_1972_2022['ANNEE'].min())
max_year = int(feux_1972_2022['ANNEE'].max())

if analysis_year < min_year or analysis_year > max_year:
    print(f"WARNING: Year {analysis_year} is outside the available range ({min_year}-{max_year})")
    print(f"Setting year to {max_year} (most recent available)")
    analysis_year = max_year
else:
    print(f"Selected year for analysis: {analysis_year}")
    print(f"Available year range in dataset: {min_year} to {max_year}")

# Filter for the selected year
feux_selected_year = feux_1972_2022[feux_1972_2022['ANNEE'] == analysis_year].copy()

print(f"\n{analysis_year} fire data:")
print(f"  - Total records: {len(feux_selected_year)}")
print(f"  - Geometry type: {feux_selected_year.geometry.geom_type.unique()}")

## Section 5: Analyze Selected Year Fire Data


In [ ]:
# Plot statistical distributions
print("\n" + "=" * 80)
print(f"STATISTICAL DISTRIBUTIONS FOR {analysis_year}")
print("=" * 80)

# Check which columns are available
available_cols = [col for col in ['CAUSE', 'SECTION', 'SUP_HA', 'PLAN_HA'] if col in feux_selected_year.columns]
print(f"\nAvailable columns for analysis: {available_cols}")

# Create subplots
fig, axes = plt.subplots(len(available_cols), 1, figsize=(12, 4 * len(available_cols)))
if len(available_cols) == 1:
    axes = [axes]

for idx, col in enumerate(available_cols):
    ax = axes[idx]

    if col == 'CAUSE':
        # Categorical data - bar chart
        value_counts = feux_selected_year[col].value_counts()
        ax.bar(value_counts.index, value_counts.values, color='steelblue', edgecolor='black')
        ax.set_xlabel('Cause')
        ax.set_ylabel('Count')
        ax.set_title(f'Distribution of {col} in {analysis_year}')
        ax.tick_params(axis='x', rotation=45)

        # Add value labels on bars
        for i, v in enumerate(value_counts.values):
            ax.text(i, v + 0.5, str(v), ha='center', va='bottom')
    elif col == 'SECTION':
        # Categorical data - bar chart
        value_counts = feux_selected_year[col].value_counts()
        ax.bar(value_counts.index, value_counts.values, color='steelblue', edgecolor='black')
        ax.set_xlabel('Section')
        ax.set_ylabel('Count')
        ax.set_title(f'Distribution of {col} in {analysis_year}')
        ax.tick_params(axis='x', rotation=45)

        # Add value labels on bars
        for i, v in enumerate(value_counts.values):
            ax.text(i, v + 0.5, str(v), ha='center', va='bottom')
    else:
        # Numerical data - histogram with equally spaced bins in log space
        data = feux_selected_year[col].dropna()
        if len(data) > 0:
            # Filter out zero and negative values for log scale
            data_positive = data[data > 0]
            if len(data_positive) > 0:
                # Create equally spaced bins in log space
                bins = np.logspace(np.log10(data_positive.min()), np.log10(data_positive.max()), 30)

                ax.hist(data_positive, bins=bins, color='steelblue', edgecolor='black', alpha=0.7)
                ax.set_xlabel(f'{col} (hectares) - log scale')
                ax.set_ylabel('Frequency')
                ax.set_title(f'Distribution of {col} in {analysis_year} (log scale)')
                ax.set_xscale('log')

                # Add mean and median lines
                ax.axvline(data_positive.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {data_positive.mean():.2f}')
                ax.axvline(data_positive.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {data_positive.median():.2f}')
                ax.legend()

                # Add statistics text
                stats_text = f'Min: {data_positive.min():.2f}\nMax: {data_positive.max():.2f}\nStd: {data_positive.std():.2f}'
                ax.text(0.95, 0.95, stats_text, transform=ax.transAxes,
                       verticalalignment='top', horizontalalignment='right',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
            else:
                ax.text(0.5, 0.5, 'No positive data available for log scale', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(f'Distribution of {col} in {analysis_year}')
        else:
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'Distribution of {col} in {analysis_year}')

plt.tight_layout()
plt.savefig(f'../data/fire_{analysis_year}_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nStatistical summary for {analysis_year}:")
for col in available_cols:
    print(f"\n{col}:")
    if col == 'CAUSE':
        print(feux_selected_year[col].value_counts())
    else:
        print(feux_selected_year[col].describe())

In [ ]:
## Section 6: Create Interactive Map for Selected Year

print("=" * 80)
print(f"CREATING INTERACTIVE MAP FOR {analysis_year}")
print("=" * 80)

# Reproject to WGS84 for folium
feux_selected_year_4326 = feux_selected_year.to_crs(epsg=4326)

# Calculate center of map
bounds = feux_selected_year_4326.total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

print(f"\nMap center: ({center_lat:.4f}, {center_lon:.4f})")

# Create folium map
m = folium.Map(location=[center_lat, center_lon], zoom_start=6, control_scale=True)

# Get all non-geometry columns for tooltip
tooltip_fields = [col for col in feux_selected_year_4326.columns if col != 'geometry']
print(f"\nTooltip fields: {tooltip_fields}")

# Manually build GeoJSON to avoid NumPy 2.0 issues
features = []
for idx, row in feux_selected_year_4326.iterrows():
    geom = row.geometry.__geo_interface__
    properties = {}
    for col in tooltip_fields:
        val = row[col]
        # Handle NaN values
        if pd.isna(val):
            properties[col] = 'N/A'
        else:
            properties[col] = str(val)
    
    feature = {
        "type": "Feature",
        "geometry": geom,
        "properties": properties
    }
    features.append(feature)

fire_geojson = json.dumps({"type": "FeatureCollection", "features": features})

# Add fire polygons to map
folium.GeoJson(
    fire_geojson,
    style_function=lambda x: {'color': 'red', 'weight': 1, 'fillOpacity': 0.3},
    tooltip=folium.GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_fields)
).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Add title
title_html = f'''
    <h3 align="center" style="font-size:16px"><b>Fire Data - {analysis_year}</b></h3>
    <p align="center" style="font-size:12px">Red: Fire Perimeters</p>
'''
m.get_root().html.add_child(folium.Element(title_html))

# Save map to HTML file (VS Code workaround)
map_filename = f'../data/fire_{analysis_year}_map.html'
m.save(map_filename)

# Display using IFrame (VS Code workaround)
display(IFrame(src=map_filename, width='100%', height=600))

print(f"\nInteractive map created for {analysis_year}")
print(f"  - Total fires: {len(feux_selected_year)}")
print(f"  - Map saved to: {map_filename}")
print("  - Use mouse to zoom in/out and pan around")
print("  - Click on features to see all details")